# 13. Action Approval and Automation Payload

## Goal

Require human approval before converting meeting action items into payloads that can later be sent to n8n.

## Why This Step Matters

ASR and extraction can be wrong. Automation must not notify people until a reviewer confirms the task, owner, and deadline.

## Inputs

Evidence-linked action items containing task, owner, deadline, approval state, and transcript evidence.

## Outputs

- `ready`: approved, assigned payloads safe for a future n8n webhook
- `blocked`: actions requiring approval, assignment, or correction

## Setup

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.automation.actions import prepare_action_payloads

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/lataeq/Code/Saranote


## Load Data

In [2]:
action_items = [
    {
        "task": "ส่งเอกสารงบประมาณฉบับแก้ไข",
        "owner": "ฝ่ายดำเนินงาน",
        "deadline": "วันศุกร์",
        "approved": False,
        "evidence": {"source": "transcript", "timestamp": "transcript-04"},
    },
    {
        "task": "ตรวจสอบข้อมูล",
        "owner": "ทุกฝ่าย",
        "deadline": "ก่อนส่งรายงาน",
        "approved": False,
        "evidence": {"source": "transcript", "timestamp": "transcript-05 - transcript-06"},
    },
    {
        "task": "จัดทำสรุปการประชุม",
        "owner": "",
        "deadline": None,
        "approved": False,
        "evidence": {"source": "transcript", "timestamp": "transcript-07"},
    },
]
print(pd.DataFrame(action_items)[["task", "owner", "deadline", "approved"]].to_string(index=False))

                         task          owner       deadline  approved
ส่งเอกสารงบประมาณฉบับแก้ไข ฝ่ายดำเนินงาน       วันศุกร์     False
                ตรวจสอบข้อมูล        ทุกฝ่าย ก่อนส่งรายงาน     False
       จัดทำสรุปการประชุม                         None     False


## Human Review

Simulate a reviewer approving the second task and the unassigned third task. The first task remains unapproved.

In [3]:
reviewed_items = [dict(item) for item in action_items]
reviewed_items[1]["approved"] = True
reviewed_items[2]["approved"] = True
print(pd.DataFrame(reviewed_items)[["task", "owner", "approved"]].to_string(index=False))

                         task          owner  approved
ส่งเอกสารงบประมาณฉบับแก้ไข ฝ่ายดำเนินงาน     False
                ตรวจสอบข้อมูล        ทุกฝ่าย      True
       จัดทำสรุปการประชุม                    True


## Run Experiment

In [4]:
prepared = prepare_action_payloads(reviewed_items)
print("READY PAYLOADS")
print(json.dumps(prepared["ready"], ensure_ascii=False, indent=2))

READY PAYLOADS
[
  {
    "event": "meeting.action_assigned",
    "owner": "ทุกฝ่าย",
    "task": "ตรวจสอบข้อมูล",
    "deadline": "ก่อนส่งรายงาน",
    "evidence": {
      "source": "transcript",
      "timestamp": "transcript-05 - transcript-06"
    }
  }
]


## Inspect Blocked Actions

In [5]:
blocked_df = pd.DataFrame(prepared["blocked"])
print(blocked_df[["task", "owner", "reason"]].to_string(index=False))

                         task          owner         reason
ส่งเอกสารงบประมาณฉบับแก้ไข ฝ่ายดำเนินงาน needs_approval
       จัดทำสรุปการประชุม     unassigned     unassigned


## Checks

In [6]:
checks = [
    ("only one payload is ready", len(prepared["ready"]) == 1),
    ("ready payload uses n8n event", prepared["ready"][0]["event"] == "meeting.action_assigned"),
    ("approved owner is preserved", prepared["ready"][0]["owner"] == "ทุกฝ่าย"),
    ("transcript evidence is preserved", prepared["ready"][0]["evidence"]["timestamp"] == "transcript-05 - transcript-06"),
    ("unapproved action is blocked", prepared["blocked"][0]["reason"] == "needs_approval"),
    ("missing owner is marked unassigned", prepared["blocked"][1]["owner"] == "unassigned"),
    ("blocked actions cannot be payloads", all(item["task"] != "ส่งเอกสารงบประมาณฉบับแก้ไข" for item in prepared["ready"])),
]
checks_df = pd.DataFrame(checks, columns=["check", "passed"])
print(checks_df.to_string(index=False))
assert checks_df["passed"].all(), "Some action approval checks failed"
print(f"Passed {checks_df['passed'].sum()}/{len(checks_df)} action approval checks")

                            check  passed
         only one payload is ready    True
      ready payload uses n8n event    True
       approved owner is preserved    True
   transcript evidence is preserved    True
       unapproved action is blocked    True
missing owner is marked unassigned    True
blocked actions cannot be payloads    True
Passed 7/7 action approval checks
